# SingleStoreSQLDatabaseRetriever

The `SingleStoreSQLDatabaseRetriever` allows you to execute SQL queries directly against a SingleStore database and retrieve results as LangChain Document objects. This integration enables AI applications to interact with structured data in real-time, making it easy to build database-aware agents and chains.

## Overview

The SingleStoreSQLDatabaseRetriever provides a seamless interface to query SingleStore databases and convert results into documents that can be used throughout LangChain applications. It supports custom row-to-document conversion, connection pooling, and integrates naturally with LangChain agents.

### Integration Details

| Class | Package | JS Support |
| :--- | :--- | :---: |
| `SingleStoreSQLDatabaseRetriever` | `langchain_singlestore` | ❌ |
| `SingleStoreSQLDatabaseChain` | `langchain_singlestore` | ❌ |

### Features
- Execute arbitrary SQL queries against SingleStore
- Automatic row-to-document conversion
- Custom row formatters for specialized requirements
- Connection pooling for performance
- Error handling and logging
- Convenience methods for common operations
- Full integration with LangChain agents and chains

## Setup

To use the `SingleStoreSQLDatabaseRetriever`, you need to install the `langchain-singlestore` package.

In [ ]:
%pip install -qU langchain-singlestore

## Initialization

To initialize `SingleStoreSQLDatabaseRetriever`, you need to provide connection parameters for the SingleStore database.

### Required Parameters:
- **host** (`str`): Hostname, IP address, or URL for the database (e.g., `localhost:3306` or `user:password@localhost:3306/database`)

### Optional Parameters:
- **user** (`str`): Database username. Can be included in the host parameter.
- **password** (`str`): Database password. Can be included in the host parameter.
- **database** (`str`): Database name. Can be included in the host parameter.

### Connection Pool Parameters:
- **pool_size** (`int`): Number of active connections in the pool. Defaults to `5`.
- **max_overflow** (`int`): Maximum connections beyond `pool_size`. Defaults to `10`.
- **timeout** (`float`): Connection timeout in seconds. Defaults to `30`.

### Custom Row Conversion:
- **row_to_document_fn** (`callable`, optional): Custom function to convert database rows to Document objects. Function signature: `(row_dict: dict, row_index: int) -> Document`

### Additional Options:
- **pure_python** (`bool`): Enables pure Python mode.
- **local_infile** (`bool`): Allows local file uploads.
- **charset** (`str`): Character set for string values.
- **ssl_key**, **ssl_cert**, **ssl_ca** (`str`): Paths to SSL files.
- **ssl_disabled** (`bool`): Disables SSL.
- **ssl_verify_cert** (`bool`): Verifies server's certificate.
- **ssl_verify_identity** (`bool`): Verifies server's identity.
- **autocommit** (`bool`): Enables autocommits.

In [ ]:
from langchain_singlestore.sql_database_retriever import SingleStoreSQLDatabaseRetriever

# Initialize retriever with connection parameters
retriever = SingleStoreSQLDatabaseRetriever(
    host="127.0.0.1:3306",
    user="root",
    password="your_password",
    database="my_database",
)

## Basic Usage

Execute a SQL query and retrieve results as documents.

In [ ]:
# Execute a query and get results as documents
docs = retriever.invoke("SELECT id, name, email FROM users LIMIT 5")

print(f"Retrieved {len(docs)} document(s)")
for i, doc in enumerate(docs):
    print(f"\n--- Document {i} ---")
    print(f"Content:\n{doc.page_content}")
    print(f"Metadata: {doc.metadata}")

## Custom Row Converter

Use a custom function to format rows into documents in a specific way.

In [ ]:
from langchain_core.documents import Document

def custom_converter(row_dict: dict, row_index: int) -> Document:
    """
    Custom converter that formats rows as simple key=value pairs.
    
    Args:
        row_dict: Dictionary representation of the database row
        row_index: Index of the row in the result set
    
    Returns:
        Document: Formatted document with custom metadata
    """
    # Create a formatted string with all fields
    fields = ", ".join(f"{k}={v}" for k, v in row_dict.items())
    content = f"Record #{row_index}: {fields}"
    
    # Add custom metadata
    metadata = {
        "record_number": row_index,
        "source": "sql_database",
        "field_count": len(row_dict),
    }
    metadata.update(row_dict)
    
    return Document(page_content=content, metadata=metadata)

# Create retriever with custom converter
custom_retriever = SingleStoreSQLDatabaseRetriever(
    host="127.0.0.1:3306",
    user="root",
    password="your_password",
    database="my_database",
    row_to_document_fn=custom_converter,
)

# Execute query with custom formatting
docs = custom_retriever.invoke("SELECT id, name FROM users LIMIT 3")
for doc in docs:
    print(f"{doc.page_content}")

## Using Convenience Methods

The `SingleStoreSQLDatabaseChain` class provides convenience methods for common operations.

### Using `from_url()`

Create a retriever from a connection URL:

In [ ]:
from langchain_singlestore.sql_database_retriever import SingleStoreSQLDatabaseChain

# Create retriever from URL
url = "root:your_password@localhost:3306/my_database"
retriever = SingleStoreSQLDatabaseChain.from_url(host=url, llm=None)

# Use the retriever
docs = retriever.invoke("SELECT COUNT(*) as total_users FROM users")
print(f"Result: {docs[0].page_content}")

### Using `query_to_document()`

Execute a one-off query without managing a retriever instance:

In [ ]:
# Execute a query directly without managing retriever lifecycle
docs = SingleStoreSQLDatabaseChain.query_to_document(
    query="SELECT id, name FROM users LIMIT 10",
    host="root:your_password@localhost:3306/my_database",
    row_limit=100,  # Automatically adds LIMIT clause
)

print(f"Retrieved {len(docs)} document(s)")
if docs:
    print(f"First result:\n{docs[0].page_content}")

## Querying with Limits

The retriever supports automatic LIMIT clause application for safety:

In [ ]:
# Query with explicit limit
docs = retriever.invoke("SELECT * FROM large_table LIMIT 5")
print(f"Retrieved {len(docs)} rows with limit")

## Error Handling

The retriever provides proper error handling for common scenarios:

In [ ]:
try:
    # Attempt to query a non-existent table
    docs = retriever.invoke("SELECT * FROM non_existent_table")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

# Proper cleanup after use
retriever.close()

## Integration with LangChain Agents

Use the retriever with LangChain agents to enable database queries in agentic workflows:

In [ ]:
from langchain.agents import Tool

# Create a tool for the agent
def query_database(query: str) -> str:
    """Execute a SQL query and return results as a formatted string."""
    retriever = SingleStoreSQLDatabaseRetriever(
        host="127.0.0.1:3306",
        user="root",
        password="your_password",
        database="my_database",
    )
    try:
        docs = retriever.invoke(query)
        results = "\n".join([doc.page_content for doc in docs])
        return results
    finally:
        retriever.close()

# Create a Tool that can be used by agents
db_tool = Tool(
    name="query_database",
    func=query_database,
    description="Execute SQL queries against the SingleStore database. Use for data retrieval and analysis.",
)

# The tool can now be used with agents
print(db_tool.name)
print(db_tool.description)

## Advanced Features

### Handling JSON Data

The retriever works seamlessly with JSON columns in SingleStore:

In [ ]:
# Query results may contain JSON data
docs = retriever.invoke("""
    SELECT id, name, properties 
    FROM products 
    WHERE properties IS NOT NULL 
    LIMIT 1
""")

if docs:
    print("Document with JSON data:")
    print(docs[0].page_content)
    print("\nMetadata:")
    print(docs[0].metadata)

### Custom Connection Pool Configuration

Configure connection pooling for specific performance requirements:

In [ ]:
# High-performance configuration for frequently accessed data
high_performance_retriever = SingleStoreSQLDatabaseRetriever(
    pool_size=10,        # More concurrent connections
    max_overflow=20,      # Allow more overflow connections
    timeout=60,           # Extended timeout for slow queries
    host="127.0.0.1:3306",
    user="root",
    password="your_password",
    database="my_database",
)

# Use the retriever
docs = high_performance_retriever.invoke("SELECT * FROM analytics_data LIMIT 10")

# Clean up
high_performance_retriever.close()